# 02 — Caso A: energia condivisa, sorgente monoenergetica

**Assunzioni:** sorgente unica e monoenergetica, direzione del neutrone nota (lungo $z$).
Il parametro condiviso è $E_n$. Per ogni evento, l'angolo vero $\theta_p$ è una nuisance e viene
marginalizzato.

Per evento, la verosimiglianza è
$$L_k(E_n) = \int d\theta\, \pi(\theta)\, N(\hat E_p; E_n\cos^2\theta, \sigma_{E_p})\,
N(\hat\theta_p; \theta, \sigma_\theta).$$
Gli eventi si combinano sommando le log-verosimiglianze, con il prior contato una sola volta:
$\log p(E_n\mid D_{1:N}) = \log\pi(E_n) + \sum_k \log L_k(E_n)$, valida se gli eventi sono
condizionatamente indipendenti dato $E_n$.

**Etichette dei risultati.** (a) fatto consolidato · (b) stima toy con assunzioni esplicite ·
(c) verificato con Monte Carlo · (d) ipotesi da testare.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid
from scipy.stats import kstest, norm

from riptide_toy.constants import SEED, SIGMA_EP, SIGMA_THETA, EN_MIN, EN_MAX

plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(SEED)
from riptide_toy import forward_model, grids, kinematics, posterior_A, priors, validate
from riptide_toy.combine import combine_loglik, expected_sigma_n

en_grid = grids.energy_grid()
prior_En = priors.energy_prior(en_grid)


def mean_sigma(logpost, grid):
    w = np.exp(logpost - logpost.max())
    m = np.sum(w * grid) / np.sum(w)
    return m, np.sqrt(np.sum(w * (grid - m) ** 2) / np.sum(w))

## Caso di riferimento R1 — un singolo evento

Un evento con $\hat E_p = 1.40$ MeV e $\hat\theta_p = 0.50$ rad, risoluzioni $\sigma_{E_p} = 0.10$ MeV e
$\sigma_\theta = 0.08$ rad, prior piatto (proprio) su $E_n$.

Valore atteso per propagazione lineare **(b)**, passo per passo:

1. $\cos 0.50 = 0.8776$, quindi $\cos^2\hat\theta_p = 0.7702$;
2. stima puntuale: $\hat E_n = \hat E_p/\cos^2\hat\theta_p = 1.40/0.7702 = 1.818 \approx 1.82$ MeV;
3. derivata rispetto a $E_p$: $\partial E_n/\partial E_p = 1/\cos^2\theta = 1.298$, che moltiplicata per
   $\sigma_{E_p} = 0.10$ dà $0.130$ MeV;
4. derivata rispetto all'angolo: $E_n = E_p\cos^{-2}\theta$, quindi
   $\partial E_n/\partial\theta = 2E_p\sin\theta/\cos^3\theta = 2\hat E_n\tan\theta$.
   Con $\tan 0.50 = 0.5463$ fa $2 \times 1.818 \times 0.5463 = 1.986$ MeV/rad, che per
   $\sigma_\theta = 0.08$ dà $0.159$ MeV;
5. i due errori sono indipendenti e si sommano in quadratura:
$$\sigma_{E_n} = \sqrt{0.130^2 + 0.159^2} = \sqrt{0.0169 + 0.0253} = \sqrt{0.0422} \approx 0.21\ \text{MeV}.$$

Il contributo angolare domina: a $\theta = 0.5$ rad un errore di 0.08 rad pesa più dell'errore
del 7% su $E_p$. Il posterior esatto sulla griglia deve dare $E_n = 1.82 \pm 0.21$ MeV. È asimmetrico
(coda verso $E_n$ alti, dove $\cos^2\theta$ è piccolo), quindi media e MAP differiscono di qualche
centesimo.

In [ ]:
c2 = np.cos(0.50) ** 2
En_lin = 1.40 / c2
d_Ep, d_th = SIGMA_EP / c2, 2 * En_lin * np.tan(0.50) * SIGMA_THETA
print(f"cos^2 = {c2:.4f}  En = {En_lin:.3f}  contributi {d_Ep:.3f} e {d_th:.3f}  totale {np.hypot(d_Ep, d_th):.3f} MeV")

In [ ]:
D = (np.array([1.40]), np.array([0.50]))
logpost = posterior_A.single_event_posterior(D, en_grid, prior_En)[0]
m, s = mean_sigma(logpost, en_grid)
print(f"MAP = {en_grid[np.argmax(logpost)]:.3f} MeV, media = {m:.3f}, sigma = {s:.3f} MeV   (atteso: 1.82 +- 0.21)")

w = np.exp(logpost - logpost.max())
plt.figure(figsize=(5, 3.5))
plt.plot(en_grid, w / trapezoid(w, en_grid))
plt.axvline(1.40 / np.cos(0.50) ** 2, color="k", ls=":", label=r"$\hat E_p/\cos^2\hat\theta_p$")
plt.xlim(0.5, 3.5); plt.xlabel("$E_n$ [MeV]"); plt.ylabel("posterior"); plt.legend()
plt.show()

Il posterior è asimmetrico. $E_n$ dipende da $\theta$ attraverso $1/\cos^2\theta$, che è
convessa: l'incertezza angolare allarga soprattutto la coda verso le alte energie.

## Caso di riferimento R2 — combinare due eventi

Due eventi con verosimiglianze circa gaussiane, $N(2.70, 0.45)$ e $N(2.35, 0.20)$ MeV. Sommare due
log-verosimiglianze gaussiane dà ancora una gaussiana, con peso $w = 1/\sigma^2$ **(a)**:

1. pesi: $w_1 = 1/0.45^2 = 4.94$, $w_2 = 1/0.20^2 = 25.0$, somma $29.94$;
2. media pesata: $\hat E_n = (2.70 \times 4.94 + 2.35 \times 25.0)/29.94 = (13.33 + 58.75)/29.94 = 2.41$, cioè $\approx 2.40$ MeV;
3. larghezza: $\sigma = 1/\sqrt{29.94} = 0.183 \approx 0.18$ MeV.

Il secondo evento, più preciso, pesa cinque volte il primo: la combinazione sta vicino a 2.35 ed
è più stretta di entrambi.

In [ ]:
grid = np.linspace(0.5, 6.0, 2000)
ll = np.stack([-0.5 * ((grid - 2.70) / 0.45) ** 2, -0.5 * ((grid - 2.35) / 0.20) ** 2])
combined = combine_loglik(ll, np.zeros_like(grid))
m, s = mean_sigma(combined, grid)
print(f"combinato: {m:.3f} +- {s:.3f} MeV   (atteso: 2.40 +- 0.18)")
wm = (2.70 / 0.45**2 + 2.35 / 0.20**2) / (1 / 0.45**2 + 1 / 0.20**2)
print(f"media pesata analitica: {wm:.3f} +- {(1 / 0.45**2 + 1 / 0.20**2) ** -0.5:.3f}")

plt.figure(figsize=(5, 3.5))
for row, lab in zip(ll, ("evento 1", "evento 2")):
    plt.plot(grid, np.exp(row) / trapezoid(np.exp(row), grid), label=lab)
w = np.exp(combined - combined.max())
plt.plot(grid, w / trapezoid(w, grid), "k", lw=2, label="combinato")
plt.xlim(1, 4); plt.xlabel("$E_n$ [MeV]"); plt.legend(); plt.show()

## Contrazione del posterior con N

Si simulano eventi con $E_n = 2.5$ MeV e $\theta_p \sim U(0,\pi/2)$, coerente con il prior uniforme su
$\theta$ della verosimiglianza del Caso A. Poi si combinano i primi $N$ eventi.

**Perché $1/\sqrt N$.** Se gli eventi sono indipendenti, le curvature delle log-verosimiglianze si
sommano: $1/\sigma_N^2 = \sum_k 1/\sigma_k^2 \approx N/\sigma_1^2$, quindi $\sigma_N = \sigma_1/\sqrt N$.
Per esempio, con una larghezza di singolo evento $\sigma_1 = 0.5$ MeV servono $N = 100$ eventi per
arrivare a $0.05$ MeV, e $N = 10\,000$ per $0.005$ MeV: per guadagnare un fattore 10 serve un fattore
100 in statistica **(c)**.

In [ ]:
EN_TRUE = 2.5
n_max = 5000
theta_true = rng.uniform(0, np.pi / 2, n_max)
Ep_true = kinematics.proton_energy(EN_TRUE, theta_true)
D = forward_model.measure(Ep_true, theta_true, SIGMA_EP, SIGMA_THETA, rng)

th_grid = grids.theta_p_grid()
log_prior_theta = np.full(th_grid.shape, -np.log(th_grid.size))
log_prior_En = np.log(prior_En)
ll = forward_model.loglik(D, en_grid, th_grid, SIGMA_EP, SIGMA_THETA, log_prior_theta, chunk_size=100)

N_VALUES = [1, 2, 10, 100]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sig, off = [], []
for n in N_VALUES:
    c = combine_loglik(ll[:n], log_prior_En)
    m, s = mean_sigma(c, en_grid)
    sig.append(s); off.append(abs(m - EN_TRUE))
    w = np.exp(c - c.max())
    axes[0].plot(en_grid, w / trapezoid(w, en_grid), label=f"N={n}")
axes[0].axvline(EN_TRUE, color="k", ls=":")
axes[0].set_xlim(1.5, 4); axes[0].set_xlabel("$E_n$ [MeV]"); axes[0].legend()
axes[1].loglog(N_VALUES, sig, "o-", label="sigma posteriore")
axes[1].loglog(N_VALUES, off, "s-", label="|media - vero|")
axes[1].loglog(N_VALUES, expected_sigma_n(sig[0], np.array(N_VALUES)), "k--", label=r"$\sigma_1/\sqrt{N}$")
axes[1].set_xlabel("N"); axes[1].legend()
plt.tight_layout(); plt.show()
for n, s, o in zip(N_VALUES, sig, off):
    print(f"N={n:4d}  sigma={s:.4f}  sigma_1/sqrt(N)={sig[0] / np.sqrt(n):.4f}  |offset|={o:.4f}")

### Un sistematico comune non si media via

Si ripete l'esercizio con un errore di scala del 2% su $E_p$, comune a tutti gli eventi. La
larghezza statistica continua a scendere come $1/\sqrt N$, ma l'offset si ferma su un plateau:
un errore condiviso da tutti gli eventi non si riduce combinandoli. Nella checklist di validazione
un plateau nella contrazione è il segnale di un sistematico condiviso **(c)**.

In [ ]:
D_shift = forward_model.measure(1.02 * Ep_true, theta_true, SIGMA_EP, SIGMA_THETA, rng)
ll_shift = forward_model.loglik(D_shift, en_grid, th_grid, SIGMA_EP, SIGMA_THETA, log_prior_theta, chunk_size=100)
N_SHIFT = [1, 2, 10, 100, 1000, 5000]
res = np.array([mean_sigma(combine_loglik(ll_shift[:n], log_prior_En), en_grid) for n in N_SHIFT])

plt.figure(figsize=(5.5, 4))
plt.loglog(N_SHIFT, res[:, 1], "o-", label="sigma posteriore")
plt.loglog(N_SHIFT, np.abs(res[:, 0] - EN_TRUE), "s-", label="|media - vero|")
plt.loglog(N_SHIFT, res[0, 1] / np.sqrt(N_SHIFT), "k--", label=r"$\sigma_1/\sqrt{N}$")
plt.xlabel("N"); plt.legend(); plt.title("shift sistematico 2% su $E_p$"); plt.show()
for n, (m, s) in zip(N_SHIFT, res):
    print(f"N={n:5d}  sigma={s:.4f}  |offset|={abs(m - EN_TRUE):.4f}")

L'offset si ferma attorno a 0.05 MeV, cioè il 2% di $E_n$: $0.02 \times 2.5 = 0.05$ MeV. La larghezza
statistica invece continua a scendere. Una stima d'ordine del punto in cui le due curve si incrociano viene da
$\sigma_1/\sqrt N = 0.05$, cioè $N^\ast = (\sigma_1/0.05)^2$; oltre $N^\ast$ aggiungere eventi non
migliora la stima, perché domina il sistematico. Qui $\sigma_1 \approx 0.2$ MeV dà
$N^\ast \approx 17$; nella figura l'incrocio cade fra N = 10 e 100, perché a N piccolo la contrazione
è più lenta di $1/\sqrt N$. Nella contrazione senza sistematico, invece, a N molto
piccolo il rapporto con $\sigma_1/\sqrt N$ non è esatto: il posterior di pochi eventi non è gaussiano
e la scala $1/\sqrt N$ vale solo in regime asintotico.

In [ ]:
print(f"sigma_1 = {res[0, 1]:.3f} MeV  ->  N* = (sigma_1 / 0.05)^2 = {(res[0, 1] / 0.05) ** 2:.0f} eventi")

## Caso di riferimento R3 — la checklist smaschera una ricostruzione difettosa

Si costruisce una ricostruzione volutamente sbagliata:
$\hat E = 1.03\,E + 0.05 + \varepsilon$, con $\varepsilon\sim N(0, 0.08E)$ e un'incertezza dichiarata
più stretta del 30% ($\hat\sigma = 0.7\cdot 0.08E$). Su 20 000 eventi con $E\sim U(1,5)$ ci si aspetta:

- bias $0.03E + 0.05$, cioè da ≈ 0.08 MeV nel primo bin a ≈ 0.20 MeV nell'ultimo;
- pull con larghezza $1/0.7 \approx 1.4$ e media ≈ 0.9 (il bias diviso per $\hat\sigma$);
- copertura dell'intervallo al 68% nominale che scende a ≈ 45%, per l'effetto combinato di bias
  e intervalli troppo stretti.

In [ ]:
def faulty(truth, rng):
    true_sigma = 0.08 * truth
    est = 1.03 * truth + 0.05 + rng.normal(0.0, true_sigma)
    sigma_hat = 0.7 * true_sigma
    z = norm.ppf(0.5 + np.array([0.68]) / 2)
    hw = z[None, :] * sigma_hat[:, None]
    return est, sigma_hat, np.stack([est[:, None] - hw, est[:, None] + hw], axis=-1)

truth = rng.uniform(1.0, 5.0, 20_000)
r = validate.run_checklist(faulty, truth, levels=np.array([0.68]), rng=rng)
print(f"bias primo/ultimo bin: {r['bias'][0]:.3f} / {r['bias'][-1]:.3f} MeV   (atteso 0.08 / 0.20)")
print(f"pull: media {r['pull_mean']:.2f}, larghezza {r['pull_width']:.2f}   (atteso 0.9 / 1.4)")
print(f"coverage 68% nominale: {r['coverage'][0]:.3f}   (atteso ~0.45)")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(r["bias_centers"], r["bias"], "o-"); axes[0].axhline(0, color="k", lw=0.8)
axes[0].set_xlabel("E vero [MeV]"); axes[0].set_ylabel("bias [MeV]")
est, sh, _ = faulty(truth, rng)
x = np.linspace(-5, 5, 200)
axes[1].hist(validate.pull_histogram(truth, est, sh), bins=80, range=(-5, 5), density=True, alpha=0.7)
axes[1].plot(x, norm.pdf(x), "k--", label="N(0,1)"); axes[1].set_xlabel("pull"); axes[1].legend()
plt.tight_layout(); plt.show()

I numeri riproducono i valori attesi. Il bias dell'ultimo bin può uscire un po' sotto 0.20 MeV:
è una fluttuazione statistica del bin più popolato a energie alte, dove lo scatter vale $0.08E$.

## Checklist sulla ricostruzione reale del Caso A

La stessa checklist si applica al posterior del Caso A. La stima è la media del posterior di singolo
evento, l'incertezza la sua deviazione standard, e gli intervalli credibili sono letti direttamente
dalla griglia **(c)**. Si confrontano due generatori della verità, con lo stesso posterior (prior
piatto su $[0.5, 6]$ MeV):

* **(i)** $E_n \sim U(0.5, 6)$: la verità è estratta dal prior stesso;
* **(ii)** $E_n \sim U(1, 5)$: una sotto-popolazione lontana dai bordi del dominio.

**Esempio guidato: perché con (i) la coverage è esatta (a).** Sia $C(D)$ l'intervallo credibile al
livello $L$ calcolato dai dati $D$. Se la verità $E$ è estratta dal prior e i dati dal modello, la
coppia $(E, D)$ ha distribuzione congiunta $\pi(E)\,p(D\mid E) = p(D)\,p(E\mid D)$. Allora
$$P\big(E \in C(D)\big) = \int p(D)\, \underbrace{P\big(E\in C(D)\mid D\big)}_{=L\ \text{per costruzione}}\, dD = L.$$
La coverage è esatta **in media sul prior**, non per ogni valore di $E$: a $E$ fissato la media
posteriore è tirata verso il centro del prior (*shrinkage*). Gli eventi con $E_n$ vicino a 0.5 MeV
sono sovrastimati, quelli vicino a 6 MeV sottostimati, e lì gli intervalli mancano più spesso la
verità. Con il generatore (ii) si tolgono proprio quei bordi: la coverage media sale sopra il
nominale e il pull si stringe. Non è un difetto del posterior, ma una media fatta su una
popolazione diversa dal prior.

In [ ]:
LEVELS = np.array([0.68, 0.90, 0.95])

def reco_A(truth, rng):
    th = rng.uniform(0, np.pi / 2, truth.size)
    D = forward_model.measure(kinematics.proton_energy(truth, th), th, SIGMA_EP, SIGMA_THETA, rng)
    # a blocchi di 50 eventi per contenere la memoria
    lp = np.concatenate([posterior_A.single_event_posterior((D[0][i:i + 50], D[1][i:i + 50]), en_grid, prior_En)
                         for i in range(0, truth.size, 50)])
    est, sig = validate.posterior_mean_std(lp, en_grid)
    return est, sig, validate.credible_interval(lp, en_grid, LEVELS)

N_CHECK = 5000
print(f"errore binomiale con {N_CHECK} eventi: {np.round(np.sqrt(LEVELS * (1 - LEVELS) / N_CHECK), 4)}")
fig, ax = plt.subplots(figsize=(5.5, 3.5))
for name, (lo, hi) in {"(i) U(0.5, 6) = prior": (EN_MIN, EN_MAX), "(ii) U(1, 5)": (1.0, 5.0)}.items():
    rA = validate.run_checklist(reco_A, rng.uniform(lo, hi, N_CHECK), levels=LEVELS, n_bins=5, rng=rng)
    print(f"{name:22s} pull: media {rA['pull_mean']:+.3f}, larghezza {rA['pull_width']:.3f} | "
          f"coverage {np.round(rA['coverage'], 3)}")
    ax.plot(rA["bias_centers"], rA["bias"], "o-", label=name)
ax.axhline(0, color="k", lw=0.8); ax.legend()
ax.set_xlabel("$E_n$ vero [MeV]"); ax.set_ylabel("bias della media posteriore [MeV]"); plt.show()

Lettura. Con `scripts/caso_A_checklist.py` (20 000 eventi per generatore, errore binomiale ±0.003 al 68%):

| generatore | pull media | pull larghezza | coverage 68 / 90 / 95 |
|---|---|---|---|
| (i) $U(0.5, 6)$ = prior | +0.003 | 0.998 | 0.683 / 0.899 / 0.949 |
| (ii) $U(1, 5)$ | +0.115 | 0.927 | 0.735 / 0.925 / 0.964 |

* con il prior uguale al generatore la coverage è nominale entro un errore binomiale e il pull è
  $N(0,1)$: il posterior del Caso A è **calibrato (c)**;
* il bias per bin è lo shrinkage previsto: +0.51 MeV nel bin a 1.05 MeV, circa zero al centro
  (+0.08 a 3.25 MeV), −0.69 MeV nel bin a 5.45 MeV. È grande perché un singolo evento informa poco:
  $E_p\sim U(0, E_n)$ e l'angolo pesa molto;
* con (ii) la coverage sale sopra il nominale per l'effetto descritto sopra. I numeri di questo
  notebook (5000 eventi) devono coincidere con questi entro l'errore binomiale.